In [ ]:
!pip install openai langchain langchain-openai
!pip install langchain-community langchain-text-splitters
!pip install faiss-cpu pypdf reportlab koreanize-matplotlib

In [ ]:
# ============================================================
# 실습 3-2. RFP 기반 연구과제 제안서 자동 작성
# 입력: pdfs/연구과제_제안서_초안.pdf (정부 RFP 공고문)
# 출력: 연구과제_제안서_초안_RAG생성.pdf
# ============================================================

!pip install -q openai langchain langchain-openai langchain-community \
               langchain-text-splitters faiss-cpu pypdf \
               reportlab koreanize-matplotlib

import os
import urllib.request
import urllib.parse
import matplotlib.pyplot as plt
import koreanize_matplotlib
plt.rcParams['axes.unicode_minus'] = False

# ── API 키 설정 ──────────────────────────────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY").strip()
    print("Colab Secrets에서 API 키 로드 완료")
except Exception:
    pass
# os.environ["OPENAI_API_KEY"] = "sk-..."

if not os.environ.get("OPENAI_API_KEY"):
    raise ValueError(
        "API 키가 설정되지 않았습니다.\n"
        "방법 A: 왼쪽 사이드바 열쇠 아이콘 -> 'OPENAI_API_KEY' 등록\n"
        "방법 B: os.environ['OPENAI_API_KEY'] = 'sk-...' 주석 해제"
    )

from openai import OpenAI
from langchain_community.document_loaders import PyPDFLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

client     = OpenAI()
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
print(f"API 키 확인: sk-...{os.environ['OPENAI_API_KEY'][-4:]}")


# ============================================================
# [실습 변수]
# ============================================================
RFP_CHUNK_SIZE    = 300
RFP_CHUNK_OVERLAP = 30
RFP_K             = 4
DRAFT_TEMPERATURE = 0.4
VERIFY_TEMPERATURE = 0.1
OUTPUT_PDF        = "연구과제_제안서_초안_RAG생성.pdf"   # 입력 파일과 다른 이름

RFP_QUERIES = {
    "사업_목적":     "이 연구과제의 사업 목적은 무엇인가?",
    "지원_규모":     "연구비 규모와 연구 기간은 얼마인가?",
    "신청_자격":     "신청 자격 요건은 무엇인가?",
    "필수_제출항목": "필수 제출 항목은 무엇인가?",
    "평가_기준":     "평가 기준과 배점은 어떻게 되는가?",
    "주요_요구사항": "기술적 요구사항이나 특별 조건은 무엇인가?",
}

SAMPLE_RFP = """
[연구과제 공모 요강 - 인공지능 기반 과학기술 문헌 분석 시스템 개발]

1. 사업 목적
과학기술 분야 연구자의 문헌 검토 업무 효율화를 위한
인공지능 기반 자동 분석 시스템을 개발한다.

2. 지원 규모
- 연구비: 연 2억원 이내
- 연구 기간: 2년 (2025년 6월 ~ 2027년 5월)
- 지원 과제 수: 3개 내외

3. 신청 자격
- 국내 대학, 출연연구기관, 기업부설연구소
- 최근 3년 내 NLP 또는 정보검색 분야 SCI(E) 논문 2편 이상

4. 필수 제출 항목
- 연구 목적 및 필요성 (A4 2페이지 이내)
- 연구 방법론 및 추진 체계 (A4 3페이지 이내)
- 기대 효과 및 활용 방안 (A4 1페이지 이내)
- 연구팀 구성 및 역할 분담
- 연도별 연구비 편성 계획

5. 평가 기준
- 연구의 창의성 및 도전성: 30점
- 연구팀 역량 및 전문성: 25점
- 연구 방법론의 타당성: 25점
- 기대 효과 및 실용성: 20점

6. 주요 요구사항
- 한국어 과학기술 문헌 처리 특화 모델 개발 필수
- 오픈소스 기반 시스템 구축 권장
- 연구 결과물 공개 의무 (GitHub 등)
- 중간 점검 보고서 연 1회 제출

7. 제출 기한
2025년 4월 30일 오후 6시
"""


# ── 공통 함수 ─────────────────────────────────────────────────
def section(title):
    print(f"\n{'='*60}")
    print(f"  {title}")
    print(f"{'='*60}")

def build_rag_chain(vectorstore, k=RFP_K, strict=True):
    retriever = vectorstore.as_retriever(search_kwargs={"k": k})
    strict_tmpl = (
        "반드시 아래 제공된 문서 내용만을 근거로 답하세요.\n"
        "문서에 없는 내용은 '제공된 문서에서 확인할 수 없습니다'라고 답하세요.\n"
        "답변 끝에 참고한 조항을 명시하세요.\n\n"
        "문서 내용:\n{context}\n\n"
        "질문: {question}\n답변:"
    )
    general_tmpl = (
        "아래 문서를 참고하여 질문에 답하세요.\n\n"
        "문서:\n{context}\n\n"
        "질문: {question}\n답변:"
    )
    prompt = PromptTemplate.from_template(
        strict_tmpl if strict else general_tmpl
    )
    llm   = ChatOpenAI(model="gpt-4o-mini", temperature=VERIFY_TEMPERATURE)
    fmt   = lambda docs: "\n\n".join(d.page_content for d in docs)
    chain = (
        {"context": retriever | fmt, "question": RunnablePassthrough()}
        | prompt | llm | StrOutputParser()
    )
    return chain, retriever

def rag_query(question, chain, retriever, label=""):
    answer  = chain.invoke(question)
    sources = retriever.invoke(question)
    if label:
        print(f"\n[{label}]  질문: {question}")
        print(f"{'─'*55}")
        print(f"답변:\n{answer}")
        print(f"{'─'*55}")
        print(f"참조 청크 ({len(sources)}개):")
        for i, d in enumerate(sources, 1):
            pg = d.metadata.get("page", "?")
            print(f"  [{i}] p.{pg}  |  {d.page_content[:70]}...")
    return answer, sources


# ── 단계 1: RFP PDF 다운로드 ─────────────────────────────────
section("단계 1. RFP PDF 다운로드")

GITHUB_RAW = (
    "https://raw.githubusercontent.com/"
    "hongsukyi/Lectures/main/04_Rag/pdfs"
)
SAVE_DIR     = "/content/pdfs"
RFP_FILENAME = "연구과제_제안서_초안.pdf"   # 정부 RFP 공고문
RFP_PDF_PATH = f"{SAVE_DIR}/{RFP_FILENAME}"

os.makedirs(SAVE_DIR, exist_ok=True)

# PDF 다운로드 시도
if os.path.exists(RFP_PDF_PATH):
    print(f"이미 존재: {RFP_FILENAME}")
    download_ok = True
else:
    url = f"{GITHUB_RAW}/{urllib.parse.quote(RFP_FILENAME)}"
    try:
        urllib.request.urlretrieve(url, RFP_PDF_PATH)
        size_kb = os.path.getsize(RFP_PDF_PATH) / 1024
        print(f"다운로드 완료: {RFP_FILENAME}  ({size_kb:.0f} KB)")
        download_ok = True
    except Exception as e:
        print(f"다운로드 실패: {e}")
        download_ok = False

# PDF 로드 또는 샘플 텍스트로 대체
if download_ok and os.path.exists(RFP_PDF_PATH):
    loader   = PyPDFLoader(RFP_PDF_PATH)
    rfp_docs = loader.load()
    rfp_src  = RFP_FILENAME
    print(f"RFP 로드 완료: {len(rfp_docs)} 페이지")
else:
    print("샘플 RFP 텍스트로 진행합니다.")
    with open("sample_rfp.txt", "w", encoding="utf-8") as f:
        f.write(SAMPLE_RFP)
    loader   = TextLoader("sample_rfp.txt", encoding="utf-8")
    rfp_docs = loader.load()
    rfp_src  = "샘플 RFP"
    print(f"샘플 RFP 로드 완료")


# ── 단계 2: RFP 벡터 저장소 구성 ────────────────────────────
section("단계 2. RFP 벡터 저장소 구성")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=RFP_CHUNK_SIZE,
    chunk_overlap=RFP_CHUNK_OVERLAP,
    separators=["\n\n", "\n", ". ", " "],
)
rfp_chunks = splitter.split_documents(rfp_docs)
print(f"청크 수: {len(rfp_chunks)}  (chunk_size={RFP_CHUNK_SIZE})")

vs_rfp              = FAISS.from_documents(rfp_chunks, embeddings)
chain_rfp, ret_rfp  = build_rag_chain(vs_rfp, k=RFP_K, strict=True)
print("벡터 저장소 구성 완료")


# ── 단계 3: RFP 핵심 요건 자동 추출 ─────────────────────────
section("단계 3. RFP 핵심 요건 자동 추출 (RAG)")

rfp_extracted = {}
for key, question in RFP_QUERIES.items():
    answer, _ = rag_query(question, chain_rfp, ret_rfp,
                          label=key.replace("_", " "))
    rfp_extracted[key] = answer


# ── 단계 4: 제안서 섹션 초안 생성 (LLM) ─────────────────────
section("단계 4. 제안서 섹션 초안 생성 (LLM)")

def generate_section(section_name, instruction, rfp_context,
                     temperature=DRAFT_TEMPERATURE):
    system = (
        f"당신은 국가 연구과제 제안서 작성 전문가입니다.\n"
        f"아래 RFP 요건을 반드시 충족하는 {section_name}을 작성하세요.\n"
        f"조건:\n"
        f"- RFP에서 요구하는 항목을 빠짐없이 포함하세요.\n"
        f"- 수치나 연구진 정보는 [기관명], [연구책임자], [금액] 형태로 표시하세요.\n\n"
        f"RFP 관련 요건:\n{rfp_context}"
    )
    resp = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system},
            {"role": "user",   "content": instruction},
        ],
        temperature=temperature,
        max_tokens=600,
    )
    return resp.choices[0].message.content

proposal_sections = {}

print("\n-- 섹션 1: 연구 목적 및 필요성 --")
proposal_sections["연구 목적 및 필요성"] = generate_section(
    section_name="연구 목적 및 필요성",
    instruction="A4 2페이지 분량의 연구 목적 및 필요성을 작성하세요.",
    rfp_context=(
        rfp_extracted.get("사업_목적", "") + "\n" +
        rfp_extracted.get("주요_요구사항", "")
    ),
)
print(proposal_sections["연구 목적 및 필요성"][:400] + "...")

print("\n-- 섹션 2: 연구 방법론 및 추진 체계 --")
proposal_sections["연구 방법론 및 추진 체계"] = generate_section(
    section_name="연구 방법론 및 추진 체계",
    instruction="A4 3페이지 분량의 연구 방법론과 연도별 추진 체계를 작성하세요.",
    rfp_context=(
        rfp_extracted.get("주요_요구사항", "") + "\n" +
        rfp_extracted.get("평가_기준", "")
    ),
)
print(proposal_sections["연구 방법론 및 추진 체계"][:400] + "...")

print("\n-- 섹션 3: 기대 효과 및 활용 방안 --")
proposal_sections["기대 효과 및 활용 방안"] = generate_section(
    section_name="기대 효과 및 활용 방안",
    instruction="A4 1페이지 분량의 기대 효과와 실용적 활용 방안을 작성하세요.",
    rfp_context=(
        rfp_extracted.get("사업_목적", "") + "\n" +
        rfp_extracted.get("평가_기준", "")
    ),
    temperature=0.5,
)
print(proposal_sections["기대 효과 및 활용 방안"][:400] + "...")


# ── 단계 5: 초안 RFP 요건 충족 검증 ─────────────────────────
section("단계 5. 제안서 초안 RFP 요건 충족 검증 (RAG)")

def verify_against_rfp(section_name, draft_text, retriever):
    prompt = PromptTemplate.from_template(
        "아래 RFP 요건과 제안서 초안을 비교하여 검증하세요.\n\n"
        "RFP 요건:\n{context}\n\n"
        "제안서 초안 ({section}):\n{draft}\n\n"
        "검증 결과를 아래 형식으로 답하세요:\n"
        "[충족 항목] ...\n"
        "[미충족 항목] ...\n"
        "[보완 권고] ..."
    )
    llm     = ChatOpenAI(model="gpt-4o-mini", temperature=VERIFY_TEMPERATURE)
    sources = retriever.invoke(f"{section_name} 요건 충족 확인")
    context = "\n\n".join(d.page_content for d in sources)
    result  = llm.invoke(
        prompt.format(context=context, section=section_name,
                      draft=draft_text[:500])
    )
    return result.content

for sec_name, draft in proposal_sections.items():
    print(f"\n[{sec_name} 검증]")
    print(verify_against_rfp(sec_name, draft, ret_rfp))


# ── 단계 6: 한글 PDF 생성 ────────────────────────────────────
section("단계 6. 한글 PDF 자동 생성")

from reportlab.lib.pagesizes import A4
from reportlab.lib.units import mm
from reportlab.platypus import (SimpleDocTemplate, Paragraph, Spacer,
                                 HRFlowable, PageBreak)
from reportlab.pdfbase import pdfmetrics
from reportlab.pdfbase.ttfonts import TTFont
from reportlab.lib.enums import TA_CENTER, TA_JUSTIFY
from reportlab.lib.styles import ParagraphStyle

urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Regular.ttf",
    "NanumGothic.ttf"
)
urllib.request.urlretrieve(
    "https://github.com/google/fonts/raw/main/ofl/nanumgothic/NanumGothic-Bold.ttf",
    "NanumGothicBold.ttf"
)
pdfmetrics.registerFont(TTFont("NanumGothic",     "NanumGothic.ttf"))
pdfmetrics.registerFont(TTFont("NanumGothicBold", "NanumGothicBold.ttf"))
print("한글 폰트 등록 완료")

s_title    = ParagraphStyle("T",  fontName="NanumGothicBold", fontSize=18,
                             alignment=TA_CENTER, spaceAfter=8, leading=26)
s_subtitle = ParagraphStyle("ST", fontName="NanumGothic",     fontSize=11,
                             alignment=TA_CENTER, spaceAfter=20,
                             textColor="#555555")
s_h1       = ParagraphStyle("H1", fontName="NanumGothicBold", fontSize=13,
                             spaceBefore=14, spaceAfter=6, leading=20)
s_body     = ParagraphStyle("B",  fontName="NanumGothic",     fontSize=10,
                             leading=18, spaceAfter=6, alignment=TA_JUSTIFY)
s_note     = ParagraphStyle("N",  fontName="NanumGothic",     fontSize=9,
                             leading=14, textColor="#888888")

def to_paragraphs(text, style):
    result = []
    for line in text.split("\n"):
        line = line.strip()
        if not line:
            result.append(Spacer(1, 4))
        else:
            line = (line.replace("&", "&amp;")
                        .replace("<", "&lt;")
                        .replace(">", "&gt;"))
            result.append(Paragraph(line, style))
    return result

doc   = SimpleDocTemplate(
    OUTPUT_PDF, pagesize=A4,
    leftMargin=25*mm, rightMargin=25*mm,
    topMargin=20*mm, bottomMargin=20*mm,
)
story = []

# 표지
story.append(Spacer(1, 40*mm))
story.append(Paragraph("연구과제 제안서", s_title))
story.append(Paragraph("(AI 자동 생성 초안 - 전문가 검토 필요)", s_subtitle))
story.append(HRFlowable(width="100%", thickness=1, color="#CCCCCC"))
story.append(Spacer(1, 10*mm))
story.append(Paragraph(f"생성 기반: {rfp_src}", s_note))
story.append(Paragraph("작성 방식: RAG(RFP 요건 추출) + LLM(초안 생성)", s_note))
story.append(Paragraph(f"출력 파일: {OUTPUT_PDF}", s_note))
story.append(Paragraph("[주의] 예산, 연구진, 수치는 반드시 직접 수정하세요.", s_note))
story.append(PageBreak())

# RFP 요건 요약
story.append(Paragraph("1. RFP 핵심 요건 요약 (RAG 추출)", s_h1))
story.append(HRFlowable(width="100%", thickness=0.5, color="#DDDDDD"))
story.append(Spacer(1, 4))
for key, value in rfp_extracted.items():
    story.append(Paragraph(f"[{key.replace('_', ' ')}]", s_h1))
    story += to_paragraphs(value, s_body)
    story.append(Spacer(1, 4))
story.append(PageBreak())

# 제안서 본문
for i, (sec_name, content) in enumerate(proposal_sections.items(), start=2):
    story.append(Paragraph(f"{i}. {sec_name}", s_h1))
    story.append(HRFlowable(width="100%", thickness=0.5, color="#DDDDDD"))
    story.append(Spacer(1, 4))
    story += to_paragraphs(content, s_body)
    story.append(PageBreak())

# 유의사항
story.append(Paragraph("작성 유의사항", s_h1))
for note in [
    "본 제안서는 AI가 자동 생성한 초안으로, 반드시 전문가 검토 후 사용하세요.",
    "[기관명], [연구책임자], [금액] 등의 표시 부분을 실제 정보로 대체하세요.",
    "예산 편성, 연구진 구성, 기관 고유 정보는 직접 작성이 필요합니다.",
    "RFP 요건 충족 여부는 단계 5 검증 결과를 참고하여 보완하세요.",
]:
    story.append(Paragraph(f"- {note}", s_body))

doc.build(story)
print(f"\nPDF 생성 완료: {OUTPUT_PDF}")
print(f"입력(RFP): {rfp_src}")
print(f"출력(초안): {OUTPUT_PDF}")

try:
    from google.colab import files
    files.download(OUTPUT_PDF)
    print("PDF 다운로드 시작")
except Exception:
    print(f"로컬 실행: {OUTPUT_PDF} 파일을 확인하세요.")


# ── 단계 7: 파이프라인 시각화 ────────────────────────────────
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(12, 3.5))
ax.set_xlim(0, 12)
ax.set_ylim(0, 4)
ax.axis("off")

stages = [
    (0.3,  "RFP PDF\n다운로드",   "#B5D4F4", "#042C53"),
    (2.3,  "RAG\n요건 추출",      "#9FE1CB", "#04342C"),
    (4.3,  "LLM\n초안 생성",      "#FAC775", "#412402"),
    (6.3,  "RAG\n요건 검증",      "#9FE1CB", "#04342C"),
    (8.3,  "PDF\n출력",           "#D3D1C7", "#2C2C2A"),
    (10.3, "전문가\n검토",         "#F5C4B3", "#4A1B0C"),
]
for x, label, bg, fg in stages:
    rect = mpatches.FancyBboxPatch(
        (x, 1.2), 1.7, 1.6,
        boxstyle="round,pad=0.05",
        facecolor=bg, edgecolor="none"
    )
    ax.add_patch(rect)
    ax.text(x + 0.85, 2.0, label, ha="center", va="center",
            fontsize=9, color=fg, fontweight="bold")
    if x < 10.3:
        ax.annotate("", xy=(x + 1.75, 2.0), xytext=(x + 1.7, 2.0),
                    arrowprops=dict(arrowstyle="->", color="#888780", lw=1.5))

ax.set_title("RFP 기반 제안서 자동 작성 파이프라인", fontsize=12, pad=10)
legend_items = [
    mpatches.Patch(facecolor="#B5D4F4", label="PDF 입력"),
    mpatches.Patch(facecolor="#9FE1CB", label="RAG"),
    mpatches.Patch(facecolor="#FAC775", label="LLM 생성"),
    mpatches.Patch(facecolor="#D3D1C7", label="코드 출력"),
    mpatches.Patch(facecolor="#F5C4B3", label="사람 필수"),
]
ax.legend(handles=legend_items, loc="lower center",
          ncol=5, fontsize=9, frameon=False,
          bbox_to_anchor=(0.5, -0.05))
plt.tight_layout()
plt.savefig("rfp_pipeline.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n종합 실습 완료.")
print(f"  입력 RFP : {rfp_src}")
print(f"  출력 초안: {OUTPUT_PDF}")
print("  RAG  - RFP 문서에서 근거 있는 요건 추출 및 교차 검증")
print("  LLM  - 추출된 요건 기반으로 창의적 초안 생성")
print("  사람 - 수치, 예산, 연구진 정보는 반드시 직접 확인")